# Оценка численности населения на основе данных реформы ЖКХ


Данные о населении — одни из самых важных в городской аналитике. Однако точной информации нет (возможно ли её получить?), и нам остаётся лишь оценивать. Один из возможных параметров для оценки МКД (многоквартирных домов) — общая жилая площадь. Зная (или рассчитав) среднюю обеспеченность мы можем понять количество человек в каждом доме


#### Реестр многоквартирных домов в рамках реформы ЖКХ

Реестр многоквартирных домов — это система, содержащая информацию о многоквартирных домах в России. Он был создан для улучшения управления жилым фондом, повышения прозрачности в сфере ЖКХ и упрощения контроля за состоянием зданий.

**Что содержится в Реестре МКД:**

- **ID дома** — уникальный идентификатор.
- **Адрес** — регион, город, улица, номер дома.
- **Характеристики дома**:
  - Год постройки, этажность, тип дома.
  - Общая площадь, жилая площадь, нежилая площадь помещений.
  - Состояние здания (аварийное или нет).
  - и другое

Эти данные мы можем так или иначе использовать для оценки численности населения в городах РФ. В отличие от некоторых других источников, реестр уже содержит координаты объектов в полях `lat` и `lon`.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2 Подготовка данных


В этом разделе будут использованы файлы из нашего репозитория, с первым документом вы уже работали, второй можно скачать по [ссылке](https://drive.google.com/drive/folders/1MkfdDBbyTiZJiztczHOmdTe2aHB-mki4?usp=sharing):

- **spb_admin.gpkg** — полигональные данные о границах районов и округов Санкт-Петербурга.  
  Источник: материалы курса «Методы пространственного анализа», НИУ ВШЭ (Р. Гончаров)

- **spb_mkd_reforma.csv** — данные о многоквартирных домах (МКД) Санкт-Петербурга.  
  Источник: [Фонд развития территорий](https://витрина.фрт.рф/opendata)

Прочитаем данные о границах округов в Санкт-Петербурге (spb_admin.gpkg):


In [ ]:
admin_okrug = gpd.read_file('../data/spb_admin.gpkg', layer='okrug')

admin_okrug.explore(tiles="cartodbpositron")

Прочитаем данные об МКД в Санкт-Петербурге и уберём строки с пустыми координатами:


In [ ]:
mkd = pd.read_csv('../data/spb_mkd_reforma.csv', index_col=0, low_memory=False)
mkd = mkd.dropna(subset=['lon', 'lat'])
mkd.head()

В наборе данных координаты уже записаны в отдельных полях `lat` и `lon`, поэтому можно сразу создать GeoDataFrame:


In [ ]:
mkd_gdf = gpd.GeoDataFrame(mkd, geometry=gpd.points_from_xy(mkd.lon, mkd.lat), crs="EPSG:4326")

#mkd_gdf.explore(tiles='cartodbpositron')

## 1. Оценка численности населения в каждом МКД

### 1.1. Проверка системы координат

Перед пространственным объединением проверим, что системы координат у обоих наборов данных совпадают:


In [ ]:
print("CRS точек МКД:", mkd_gdf.crs)
print("CRS округов:", admin_okrug.crs)

### 1.2. Пространственное объединение

Для каждого многоквартирного дома определяем округ, в котором он расположен. Используем метод `sjoin` с предикатом `within`:


In [ ]:
# Делаем пространственный join: для каждой точки подбираем тот округ, в который она попадает
#    predicate='within' означает, что точка должна лежать внутри полигона округа
mkd_district = gpd.sjoin(mkd_gdf, admin_okrug, how='left', predicate='within')


# Переименовываем столбцы, чтобы с ними было удобнее работать 

mkd_district = mkd_district.rename(columns={
    'NAME': 'okrug_name',        # новое имя для столбца "название округа"
    'Popul': 'okrug_population'    # новое имя для столбца "численность округа"
})

In [ ]:
mkd_district.head()

### 1.3. Расчёт обеспеченности жильём и оценка численности населения


Вычислим среднюю обеспеченность жилой площадью на основе данных о населении района


In [ ]:
# Группируем данные по району и считаем суммарную площадь жилых помещений
# и среднюю обеспеченность жилой площадью на одного жителя в районе
district_stat = mkd_district.groupby('okrug_name').agg(
    total_residential_area=('area_residential', 'sum')
).reset_index()

district_stat.head()


In [ ]:
# Объединяем эти данные с исходным DataFrame mkd_district по полю 'okrug_name'
mkd_district = mkd_district.merge(district_stat[['okrug_name', 'total_residential_area']], on='okrug_name', how='inner')

# Считаем обеспеченность
mkd_district['avg_area_per_person'] = mkd_district['total_residential_area'] / mkd_district['okrug_population']

mkd_district.head()

На основне оценочной обеспеченности населения жилой площадью, вычисляем количество жителей в каждом доме


In [ ]:
# Оцениваем кол-во человек на основе обеспеченности
mkd_district['estimated_population'] = mkd_district['area_residential'] / mkd_district['avg_area_per_person']

# Смотрим на результат
mkd_district.head()

## 2. Создание карты плотности населения


### 2.1. Регулярная сетка

Для визуализации плотности населения создадим регулярную сетку (fishnet) — набор ячеек одинакового размера, покрывающих территорию города:


Создаем функцию построения регулярной сетки для агрегирования данных


In [ ]:
from shapely.geometry import Polygon

def create_regular_grid(gdf, square_size):
    #вычислеяем utm зоны для набора данных 
    utm_zone = gdf.estimate_utm_crs()
    #перепроецируем набор данных
    gdf = gdf.to_crs(utm_zone)
    minX, minY, maxX, maxY = gdf.total_bounds
    
    grid_cells = []
    x, y = minX, minY

    while y <= maxY:
        while x <= maxX:
            geom = Polygon([(x, y), (x, y + square_size), (x + square_size, y + square_size), (x + square_size, y), (x, y)])
            grid_cells.append(geom)
            x += square_size
        x = minX
        y += square_size

    fishnet = gpd.GeoDataFrame(geometry=grid_cells, crs=utm_zone)
    fishnet['grid_id'] = fishnet.index
    return fishnet

Создаем сетку для Санкт-Петербурга 1км^2


In [ ]:
grid = create_regular_grid(mkd_district, 1000)

Определяем систему координат для перепроецирования данных МКД


In [ ]:
utm_crs = mkd_district.estimate_utm_crs()
mkd_district_utm = mkd_district.to_crs(utm_crs)

### 2.2. Расчёт плотности населения

Агрегируем оценки по ячейкам сетки и вычислим плотность населения (чел./км²) для каждой ячейки:


Рассчитаем плотность населения в каждой ячейке


In [ ]:
mkd_district_utm = mkd_district_utm.drop(columns=['index_right'])

# Выполняем пространственное соединение: сопоставляем каждую точку дома с тем квадратом сетки, 
# в котором она находится.  
msk_in_grid = gpd.sjoin(mkd_district_utm, grid, predicate='within')


In [ ]:
# Группируем результирующий набор msk_in_grid по полю 'id' (это идентификатор ячейки сетки), 
# и суммируем для каждой ячейки численность населения (столбец 'estimated_population').  
pop_grid = msk_in_grid.groupby('grid_id')['estimated_population'].sum()

# Преобразуем Series pop_grid в DataFrame: сбрасываем индекс, 
# даём новой колонке имя 'pop_sum' (это суммарное население в ячейке).
# Теперь pop_grid_df имеет два столбца: 'id' и 'pop_sum'.
pop_grid_df = pop_grid.reset_index(name='pop_sum')

# Объединяем исходный GeoDataFrame grid с таблицей pop_grid_df по столбцу 'id' (в сетке) и 'id_right' (в слое с перечечением).  
pop_grid_gdf = grid.merge(pop_grid_df,left_on='grid_id', right_on='grid_id', how='left')

# Вычисляем плотность населения для каждой ячейки: кол-во человек на квадратны километр
pop_grid_gdf['pop_density'] = pop_grid_gdf['pop_sum']/(pop_grid_gdf.geometry.area/1000000 )

### 2.3. Визуализация


Визуализируем результат


In [ ]:
pop_grid_gdf.explore(column='pop_density', cmap='YlGnBu', tiles='cartodbpositron', scheme='NaturalBreaks', k=5, legend=True, missing_kwds={'color': '#ffffff00','fillOpacity': 0})

## Итог

В этом разделе мы научились оценивать численность населения на основе данных об МКД:

- рассчитали среднюю обеспеченность жилой площадью на жителя по округам;
- оценили численность населения в каждом доме на основе этой обеспеченности;
- агрегировали данные по регулярной сетке и визуализировали пространственное распределение плотности населения.
